# Modeling

This notebook uses the features data from the folder data/processed and creates additional features.  

In [160]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from c08_farming_exit import config
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ydata_profiling import ProfileReport
import shap
from sklearn.ensemble import RandomForestRegressor
from econml.dml import CausalForestDML
from econml.cate_interpreter import SingleTreeCateInterpreter


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [161]:
# #In case you want to run Stata in a cell using the magic command %%stata, initialize it first!
# from c08_farming_exit.stata_utils import init_stata
# init_stata()

## 1. Import features data

In [162]:
df = pd.read_csv(config.PROCESSED_DATA_DIR / "features_data.csv")

## 2. Preparation and inputs

In [163]:
geo_cols = ['district', 'enumeration_area', 'region'] #these columns are missing for Botwana and Namibia
identification_cols = ['personal_id',  'interview_key', 'members_id']
df = df.drop(columns = geo_cols + identification_cols)

Y = 'off_farm_hours_annual_share'
df = df.dropna(subset=Y) # can't train on unlabeled rows

T_input = [ 'land_size_cropland_acres',
            'land_size_fallow_acres',
            'land_size_agroforestry_forestry_acres',
            'land_size_pasture_acres']

T = 'agriculture_land_acres'

T_2 =   'land_number_of_plots'

X = [   'female',
        'relation_to_head',
        'age',
        'years_of_schooling',]

W = [
        #geographics
        'country', 

        #socio-demographics
        'ethnic_group',
        'religion',
        'answered_for_self', 
    
        #land ownership
        'land_size_residential_acres',
        'land_size_lodge_camp_acres',
        'land_cropland_ownership_status',
        'land_residential_ownership_status',
        'grazing_land_ownership_status',
        'grazing_land_number_hh_sharing',
        'grazing_land_permit',
        'grazing_land_permit_price',
        'grazing_land_use_duration_in_years',

        #contracts/membership
        'livestock_contract',
        'crop_contract',
        'crop_contract_crop_type',
        'membership_farmers_group',
        'membership_agricultural_cooperative',

        #market distance and road quality
        'livestock_market_distance_in_km',
        'road_type',
        'road_condition',
        'road_distance_in_minutes',
        'market_output_distance_in_km',
        'market_input_distance_in_km',
        'market_type',

        #subsidies
        'subsidy',
        'subsidy_type_seeds',
        'subsidy_type_fertilizer',
        'subsidy_type_agro_chemicals',
        'subsidy_type_interest_free_loan',
        'subsidy_supplier_government',
        'subsidy_supplier_ngos',
        'subsidy_supplier_company',

        #assets
        'asset_value',
        'asset_diversity',
        'house_room_number',
        'house_roof_material',
        'house_wall_material',
        'house_floor_material',
        'house_water_source',
        'house_toilet_type',
        'house_energy_source',
        'house_energy_source_for_cooking',
        'house_energy_source_for_lighting',

        #mobile money and internet
        'internet_access',
        'internet_access_at_home',
        'mobile_money_access',
        
        #emotions
        'worry_about_job_loss_or_economic_livelihood',
        'life_satisfaction',
        'hh_optimism_index',
        'hh_sentiment_index',

        #add ons
        'sufficent_food_number_of_month',
        'hh_members_count',
        'beans_allocated_to_empl_occupation',
        'grazing_land_challenges_index',

        #other income/remittances
        'other_income_amount_annual',
        'remittance_amount_sent_last_12_months',

        #shock/coping
        'shock_type_affected_last_12_months_crop_failure',
        'shock_type_affected_last_12_months_drought',
        'shock_type_affected_last_12_months_floods',
        'shock_type_affected_last_12_months_illness_death',
        'shock_type_affected_last_12_months_livestock_loss',
        'shock_type_affected_last_12_months_other',
        'shock_type_affected_last_12_months_price_shock',
        'shock_coping_strategy_relatives_friends',
        'shock_coping_strategy_government',
        'shock_coping_strategy_food_reduction',
        'shock_coping_strategy_changed_cropping_practices',
        'shock_coping_strategy_more_employment',
        'shock_coping_strategy_hh_member_migration',
        'shock_coping_strategy_savings',
        'shock_coping_strategy_insurance',
        'shock_coping_strategy_credit',
        'shock_coping_strategy_sold_hh_assets',
        'shock_coping_strategy_sold_livestock',
        'shock_coping_strategy_migration',
        'shock_future_likelihood_change_income_source',

        #migration
        'migrant_last_12_months',
        'current_migrant',
        'migration_intention_next_12_months',
        
        #financing 
        'land_used_as_collateral',
        'agriculture_loan_last_5_years',
        'agriculture_loan_amount',
        'agriculture_loan_lender_bank',
        'agriculture_loan_lender_credit_union',
        'agriculture_loan_lender_private_lender',
        'agriculture_loan_lender_government',
        'agriculture_loan_lender_ngos',
 ]

df = df[[Y] + T_input + [T_2] + X + W]

## 3. Missings and outlier treatment

In [164]:
# profile_report_name = "pandas_profile_report_causal_forest"
# profile = ProfileReport(df, title="Farming Exit - Causal Forest Input")
# profile.to_file(f"../output/{profile_report_name}.html")

In [165]:
# 1. Select numeric columns (float64 and int64) and categorical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
obj_cols = df.select_dtypes(include=['object']).columns

# 2. Outlier treatment: clip values outside the 1%/99% quantiles, ignoring NaNs
for col in num_cols:
    lower = df[col].quantile(0.01)   # quantile() ignores NaNs by default
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)

# 3. Fill numeric NaNs with sentinel value
df[num_cols] = df[num_cols].fillna(99999)

# 4. Fill object NaNs/Nones with "missing"
df[obj_cols] = df[obj_cols].fillna("missing")

## 4. Create final treatment

In [166]:
#THE TARGET INPUTS SHOULD NOT CONTAIN 9999 IN CASE IF MISSINGS; BUT RATHER THE COUNTRY MEDIAN

#TARGET 1
T_input = ['land_size_cropland_acres',
           'land_size_fallow_acres',
           'land_size_agroforestry_forestry_acres',
           'land_size_pasture_acres']

for col in T_input:
    s = df[col].replace(99999, np.nan)
    medians = s.groupby(df['country']).transform('median')
    df[col] = s.fillna(medians)

df['agriculture_land_acres'] = df[T_input].sum(axis=1)
df = df.drop(columns=T_input)

#TARGET 2
s = df['land_number_of_plots'].replace(99999, np.nan)
median = s.groupby(df['country']).transform('median')
df['land_number_of_plots'] = s.fillna(median)

## 5. Causal Forest

In [167]:
df = pd.get_dummies(df, dummy_na=False)

# correct W
W = [c for c in df.columns if c not in [Y, T] + X]

In [168]:
est = CausalForestDML(
    model_y=RandomForestRegressor(
        n_estimators=200, min_samples_leaf=10, n_jobs=-1, random_state=0),
    model_t=RandomForestRegressor(
        n_estimators=200, min_samples_leaf=10, n_jobs=-1, random_state=0),
    discrete_treatment=False,
    n_estimators=2000,       # forest size for the causal forest itself
    min_samples_leaf=10,
    max_depth=None,
    cv=5,
    random_state=0,
    n_jobs=-1,
)

est.fit(Y=df[Y], T=df[T], X=df[X], W=df[W])
print("Model fit complete.")

Model fit complete.


In [170]:
# --------------------------------------------------------------------------
# 2. Tree-based visualization of treatment effects per group
# --------------------------------------------------------------------------
# SingleTreeCateInterpreter fits a shallow decision tree on X that explains
# *where in feature space* the (multi-dimensional) treatment effect is high
# or low, and shows the average effect for each leaf ("group").
intrp = SingleTreeCateInterpreter(
    include_model_uncertainty=False,
    max_depth=3,           # keep shallow so it stays readable
    min_samples_leaf=max(50, int(0.01 * len(df))),
)
intrp.interpret(est, X)

fig, ax = plt.subplots(figsize=(22, 10))
intrp.plot(
    ax=ax,
    feature_names=het_cols,
    treatment_names=treatment_cols,
    fontsize=10,
)
plt.tight_layout()
plt.savefig("cate_tree.png", dpi=150)
plt.close(fig)
print("Saved cate_tree.png")

ValueError: Expected 2D array, got 1D array instead:
array=['female' 'relation_to_head' 'age' 'years_of_schooling'].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.